In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ReduceLROnPlateau
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import os
import pickle
import json
import numpy as np

def create_mobilenetv2_model():
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)

    predictions = Dense(4, activation='softmax', kernel_regularizer=l2(0.001))(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        layer.trainable = True

    return model

def plot_learning_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    ax1.plot(history['accuracy'], label='Training Accuracy')
    ax1.plot(history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend(loc='lower right')
    ax1.grid(True)

    ax2.plot(history['loss'], label='Training Loss')
    ax2.plot(history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend(loc='upper right')
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

def convert_to_serializable(obj):
    """Convert numpy types to Python native types for JSON serialization"""
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (bytes, bytearray)):
        return obj.decode('utf-8')
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_to_serializable(i) for i in obj)
    elif hasattr(obj, 'numpy'):
        # For TensorFlow tensors
        return convert_to_serializable(obj.numpy())
    else:
        return obj

def save_history(history, filepath):
    """Save training history to a file"""
    # Convert history to a dictionary if it's a keras History object
    if not isinstance(history, dict):
        history = history.history

    # Convert all values to serializable types
    history_dict = convert_to_serializable(history)

    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(history_dict, f)
    print(f"History saved to: {filepath}")

def load_history(filepath):
    """Load training history from a file"""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            history = json.load(f)
        return history
    return None

def save_metrics(metrics, filepath):
    """Save evaluation metrics to a file"""
    # Convert metrics to serializable types
    serializable_metrics = convert_to_serializable(metrics)

    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(serializable_metrics, f)
    print(f"Metrics saved to: {filepath}")

def load_metrics(filepath):
    """Load evaluation metrics from a file"""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            metrics = json.load(f)
        return metrics
    return None

def train_mobilenetv2_model(mobilenet_model, client_data_path, model_save_path,
                            history_save_path, metrics_save_path,
                           batch_size=16, epochs=150, learning_rate=0.00005, force_retrain=False):

    # Check if model already exists
    if os.path.exists(model_save_path) and os.path.exists(history_save_path) and not force_retrain:
        print(f"Model already exists at {model_save_path}. Loading...")
        mobilenet_model = load_model(model_save_path)
        history = load_history(history_save_path)
        metrics = load_metrics(metrics_save_path)

        print("\nLoaded Metrics:")
        print("-" * 50)
        for key, value in metrics.items():
            print(f"{key}: {value}")
        print("-" * 50)

        # Plot the loaded history
        plot_learning_curves(history)

        return mobilenet_model, history

    # If model doesn't exist or force_retrain is True, train the model
    print(f"Training new model...")

    optimizer = Adam(learning_rate=learning_rate)
    mobilenet_model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    train_datagen = ImageDataGenerator(
        rescale=1.0/255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest',
        validation_split=0.2
    )

    val_datagen = ImageDataGenerator(
        rescale=1.0/255,
        validation_split=0.2
    )

    train_gen = train_datagen.flow_from_directory(
        client_data_path,
        target_size=(224, 224),
        batch_size=batch_size,
        class_mode="sparse",
        subset='training'
    )

    val_gen = val_datagen.flow_from_directory(
        client_data_path,
        target_size=(224, 224),
        batch_size=batch_size,
        class_mode="sparse",
        subset='validation'
    )

    # Only adding the learning rate scheduler callback
    callbacks = [
        # Learning rate scheduler
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=5,
            min_lr=0.00001,
            verbose=1
        )
    ]

    history = mobilenet_model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks
    )

    val_labels = val_gen.classes
    val_predictions = mobilenet_model.predict(val_gen)
    val_pred_classes = val_predictions.argmax(axis=1)

    precision = precision_score(val_labels, val_pred_classes, average="weighted")
    recall = recall_score(val_labels, val_pred_classes, average="weighted")
    accuracy = accuracy_score(val_labels, val_pred_classes)
    f1 = f1_score(val_labels, val_pred_classes, average="weighted")

    metrics = {
        "Training_Accuracy": float(history.history['accuracy'][-1]),
        "Validation_Accuracy": float(history.history['val_accuracy'][-1]),
        "Precision": float(precision),
        "Recall": float(recall),
        "F1_Score": float(f1),
        "Overall_Accuracy": float(accuracy)
    }

    print("\nFinal Metrics:")
    print("-" * 50)
    for key, value in metrics.items():
        print(f"{key}: {value}")
    print("-" * 50)

    plot_learning_curves(history.history)

    # Create directories if they don't exist
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

    # Save model
    mobilenet_model.save(model_save_path)
    print(f"Model saved to: {model_save_path}")

    # Save history
    save_history(history.history, history_save_path)

    # Save metrics
    save_metrics(metrics, metrics_save_path)

    return mobilenet_model, history.history

# Paths to each client's dataset
client1_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_1"
client2_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_2"
client3_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_3"
client4_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_4"

# Base directory for saving models and results
save_dir = "C:\\Users\\sa\\Desktop\\saved_models"
os.makedirs(save_dir, exist_ok=True)

# Paths to save models, history, and metrics
client1_model_path = os.path.join(save_dir, "client1_model.h5")
client1_history_path = os.path.join(save_dir, "client1_history.json")
client1_metrics_path = os.path.join(save_dir, "client1_metrics.json")

client2_model_path = os.path.join(save_dir, "client2_model.h5")
client2_history_path = os.path.join(save_dir, "client2_history.json")
client2_metrics_path = os.path.join(save_dir, "client2_metrics.json")

client3_model_path = os.path.join(save_dir, "client3_model.h5")
client3_history_path = os.path.join(save_dir, "client3_history.json")
client3_metrics_path = os.path.join(save_dir, "client3_metrics.json")

client4_model_path = os.path.join(save_dir, "client4_model.h5")
client4_history_path = os.path.join(save_dir, "client4_history.json")
client4_metrics_path = os.path.join(save_dir, "client4_metrics.json")

# Set force_retrain to False to use saved models if they exist
force_retrain = False

# Client 1
client1_mobilenet_model = create_mobilenetv2_model()
client1_mobilenet_model_trained, client1_history = train_mobilenetv2_model(
    client1_mobilenet_model,
    client1_data_path,
    client1_model_path,
    client1_history_path,
    client1_metrics_path,
    force_retrain=force_retrain)

# Client 2
client2_mobilenet_model = create_mobilenetv2_model()
client2_mobilenet_model_trained, client2_history = train_mobilenetv2_model(
    client2_mobilenet_model,
    client2_data_path,
    client2_model_path,
    client2_history_path,
    client2_metrics_path,
    force_retrain=force_retrain)

# Client 3
client3_mobilenet_model = create_mobilenetv2_model()
client3_mobilenet_model_trained, client3_history = train_mobilenetv2_model(
    client3_mobilenet_model,
    client3_data_path,
    client3_model_path,
    client3_history_path,
    client3_metrics_path,
    force_retrain=force_retrain)

# Client 4
client4_mobilenet_model = create_mobilenetv2_model()
client4_mobilenet_model_trained, client4_history = train_mobilenetv2_model(
    client4_mobilenet_model,
    client4_data_path,
    client4_model_path,
    client4_history_path,
    client4_metrics_path,
    force_retrain=force_retrain)

print("All models processed successfully!")


In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, confusion_matrix, classification_report, roc_curve, auc, f1_score
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
import os
import json

def load_history(filepath):
    """Load training history from a file"""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            history = json.load(f)
        return history
    return None

def save_evaluation_results(results, filepath):
    """Save evaluation results to a file"""
    # Convert numpy values to Python native types for JSON serialization
    serializable_results = {}
    for key, value in results.items():
        if isinstance(value, list):
            serializable_results[key] = [float(v) if isinstance(v, np.number) else v for v in value]
        elif isinstance(value, np.number):
            serializable_results[key] = float(value)
        else:
            serializable_results[key] = value

    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(serializable_results, f, indent=4)
    print(f"Evaluation results saved to: {filepath}")

def load_evaluation_results(filepath):
    """Load evaluation results from a file"""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            results = json.load(f)
        return results
    return None

def calculate_overall_results(client_model_paths, client_history_paths, client_data_paths,
                              evaluation_results_path, force_reevaluate=False):
    """
    Calculate and save overall results from all client models

    Parameters:
    - client_model_paths: List of paths to saved client models
    - client_history_paths: List of paths to saved client histories
    - client_data_paths: List of paths to client data directories
    - evaluation_results_path: Path to save evaluation results
    - force_reevaluate: If True, reevaluate even if results exist
    """
    # Check if evaluation results already exist
    if os.path.exists(evaluation_results_path) and not force_reevaluate:
        print(f"Evaluation results already exist at {evaluation_results_path}. Loading...")
        overall_metrics = load_evaluation_results(evaluation_results_path)

        # Display the loaded metrics
        print("\nLoaded Overall Results Across All Clients:")
        print("-" * 50)
        print(f"Average Precision: {overall_metrics['avg_precision']:.4f} ± {overall_metrics['std_precision']:.4f}")
        print(f"Average Recall: {overall_metrics['avg_recall']:.4f} ± {overall_metrics['std_recall']:.4f}")
        print(f"Average F1 Score: {overall_metrics['avg_f1_score']:.4f} ± {overall_metrics['std_f1_score']:.4f}")
        print(f"Average Training Accuracy: {overall_metrics['avg_training_accuracy']:.4f} ± {overall_metrics['std_training_accuracy']:.4f}")
        print(f"Average Validation Accuracy: {overall_metrics['avg_val_accuracy']:.4f} ± {overall_metrics['std_val_accuracy']:.4f}")
        print("-" * 50)

        # Load and display the confusion matrix
        if 'confusion_matrix' in overall_metrics:
            plt.figure(figsize=(10, 8))
            cm = np.array(overall_metrics['confusion_matrix'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title('Combined Confusion Matrix Across All Clients')
            plt.xlabel('Predicted Label')
            plt.ylabel('True Label')
            plt.show()

        # Display classification report if available
        if 'classification_report' in overall_metrics:
            print("\nClassification Report Across All Clients:")
            print("-" * 50)
            print(overall_metrics['classification_report'])

        return overall_metrics

    # If results don't exist or force_reevaluate is True, perform evaluation
    print("Calculating overall results...")

    # Load models and histories
    client_models = []
    client_histories = []

    for model_path, history_path in zip(client_model_paths, client_history_paths):
        # Load model
        if os.path.exists(model_path):
            model = load_model(model_path)
            client_models.append(model)
        else:
            raise FileNotFoundError(f"Model file not found: {model_path}")

        # Load history
        history = load_history(history_path)
        client_histories.append(history)

    overall_metrics = {
        'precision': [],
        'recall': [],
        'f1_score': [],
        'val_accuracy': []
    }

    # New metrics for training accuracy
    training_accuracy = []

    all_predictions = []
    all_true_labels = []

    # Calculate metrics for each client
    for model, data_path, history in zip(client_models, client_data_paths, client_histories):
        # Get validation metrics
        datagen = ImageDataGenerator(rescale=1.0/255)
        validation_gen = datagen.flow_from_directory(
            data_path,
            target_size=(224, 224),
            batch_size=16,
            class_mode="sparse",
            shuffle=False
        )

        val_labels = validation_gen.classes
        val_predictions = model.predict(validation_gen)
        val_pred_classes = val_predictions.argmax(axis=1)

        all_predictions.extend(val_pred_classes)
        all_true_labels.extend(val_labels)

        precision = precision_score(val_labels, val_pred_classes, average="weighted")
        recall = recall_score(val_labels, val_pred_classes, average="weighted")
        val_accuracy = accuracy_score(val_labels, val_pred_classes)
        f1 = f1_score(val_labels, val_pred_classes, average="weighted")

        overall_metrics['precision'].append(precision)
        overall_metrics['recall'].append(recall)
        overall_metrics['f1_score'].append(f1)
        overall_metrics['val_accuracy'].append(val_accuracy)

        # Get training accuracy from history
        if history and 'accuracy' in history:
            # Get the last epoch's training accuracy
            train_acc = history['accuracy'][-1]
            training_accuracy.append(train_acc)
        else:
            # Fallback if history doesn't have accuracy
            print(f"Warning: No training accuracy found in history for a client. Using validation accuracy instead.")
            training_accuracy.append(val_accuracy)

    # Calculate average metrics
    avg_precision = np.mean(overall_metrics['precision'])
    std_precision = np.std(overall_metrics['precision'])
    avg_recall = np.mean(overall_metrics['recall'])
    std_recall = np.std(overall_metrics['recall'])
    avg_f1_score = np.mean(overall_metrics['f1_score'])
    std_f1_score = np.std(overall_metrics['f1_score'])
    avg_training_accuracy = np.mean(training_accuracy)
    std_training_accuracy = np.std(training_accuracy)
    avg_val_accuracy = np.mean(overall_metrics['val_accuracy'])
    std_val_accuracy = np.std(overall_metrics['val_accuracy'])

    # Print overall results with separate training and validation accuracy
    print("\nOverall Results Across All Clients:")
    print("-" * 50)
    print(f"Average Precision: {avg_precision:.4f} ± {std_precision:.4f}")
    print(f"Average Recall: {avg_recall:.4f} ± {std_recall:.4f}")
    print(f"Average F1 Score: {avg_f1_score:.4f} ± {std_f1_score:.4f}")
    print(f"Average Training Accuracy: {avg_training_accuracy:.4f} ± {std_training_accuracy:.4f}")
    print(f"Average Validation Accuracy: {avg_val_accuracy:.4f} ± {std_val_accuracy:.4f}")
    print("-" * 50)

    # Confusion Matrix
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(all_true_labels, all_predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Combined Confusion Matrix Across All Clients')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

    # Classification Report
    class_report = classification_report(all_true_labels, all_predictions)
    print("\nClassification Report Across All Clients:")
    print("-" * 50)
    print(class_report)

    # Modified ROC Curves calculation
    plt.figure(figsize=(10, 8))
    roc_data = {}

    for i in range(4):  # Assuming 4 classes
        y_true = np.array(all_true_labels) == i
        # Concatenate predictions from all batches
        all_predictions_for_class = []
        for model, data_path in zip(client_models, client_data_paths):
            datagen = ImageDataGenerator(rescale=1.0/255)
            validation_gen = datagen.flow_from_directory(
                data_path,
                target_size=(224, 224),
                batch_size=16,
                class_mode="sparse",
                shuffle=False
            )
            predictions = model.predict(validation_gen)
            all_predictions_for_class.extend(predictions[:, i])

        y_pred = np.array(all_predictions_for_class)
        # Ensure lengths match
        min_len = min(len(y_true), len(y_pred))
        y_true = y_true[:min_len]
        y_pred = y_pred[:min_len]

        fpr, tpr, _ = roc_curve(y_true, y_pred)
        roc_auc = auc(fpr, tpr)

        # Store ROC data for saving
        roc_data[f'class_{i}'] = {
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist(),
            'auc': float(roc_auc)
        }

        plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for All Classes')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()

    # Prepare results for saving
    results_to_save = {
        'precision': overall_metrics['precision'],
        'recall': overall_metrics['recall'],
        'f1_score': overall_metrics['f1_score'],
        'val_accuracy': overall_metrics['val_accuracy'],
        'training_accuracy': training_accuracy,
        'avg_precision': avg_precision,
        'std_precision': std_precision,
        'avg_recall': avg_recall,
        'std_recall': std_recall,
        'avg_f1_score': avg_f1_score,
        'std_f1_score': std_f1_score,
        'avg_training_accuracy': avg_training_accuracy,
        'std_training_accuracy': std_training_accuracy,
        'avg_val_accuracy': avg_val_accuracy,
        'std_val_accuracy': std_val_accuracy,
        'confusion_matrix': cm.tolist(),
        'classification_report': class_report,
        'roc_data': roc_data
    }

    # Save evaluation results
    save_evaluation_results(results_to_save, evaluation_results_path)

    # Update the returned metrics to include both training and validation accuracy
    overall_metrics['training_accuracy'] = training_accuracy

    return overall_metrics

# Base directory for saved models and results
save_dir = "C:\\Users\\sa\\Desktop\\saved_models"

# Paths to saved models and histories
client1_model_path = os.path.join(save_dir, "client1_model.h5")
client1_history_path = os.path.join(save_dir, "client1_history.json")

client2_model_path = os.path.join(save_dir, "client2_model.h5")
client2_history_path = os.path.join(save_dir, "client2_history.json")

client3_model_path = os.path.join(save_dir, "client3_model.h5")
client3_history_path = os.path.join(save_dir, "client3_history.json")

client4_model_path = os.path.join(save_dir, "client4_model.h5")
client4_history_path = os.path.join(save_dir, "client4_history.json")

# Paths to client data
client1_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_1"
client2_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_2"
client3_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_3"
client4_data_path = "C:\\Users\\sa\\Desktop\\Balanced_clients1\\client_4"

# Path to save evaluation results
evaluation_results_path = os.path.join(save_dir, "overall_evaluation_results.json")

# Set force_reevaluate to False to use saved evaluation results if they exist
force_reevaluate = False

# Calculate overall results
overall_results = calculate_overall_results(
    client_model_paths=[client1_model_path, client2_model_path, client3_model_path, client4_model_path],
    client_history_paths=[client1_history_path, client2_history_path, client3_history_path, client4_history_path],
    client_data_paths=[client1_data_path, client2_data_path, client3_data_path, client4_data_path],
    evaluation_results_path=evaluation_results_path,
    force_reevaluate=force_reevaluate
)

print("Evaluation completed successfully!")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import clone_model, load_model, save_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import tensorflow as tf
import os
import json

def load_saved_models(model_paths):
    """Load saved models from the specified paths"""
    models = []
    for path in model_paths:
        if os.path.exists(path):
            print(f"Loading model from {path}")
            model = load_model(path)
            models.append(model)
        else:
            raise FileNotFoundError(f"Model file not found: {path}")
    return models

def aggregate_models_fedavg(models):
    """
    Aggregate multiple models using FedAvg principles

    In FedAvg, the aggregation step is a simple average of client models' weights.
    """
    # Use the first model as a template for the global model
    base_model = models[0]
    global_model = clone_model(base_model)

    # Get all model weights
    all_weights = [model.get_weights() for model in models]

    # Simple averaging of weights
    new_weights = []
    for weights_list in zip(*all_weights):
        new_weights.append(np.mean(weights_list, axis=0))

    # Set the averaged weights to the global model
    global_model.set_weights(new_weights)

    # Compile the global model
    global_model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return global_model

def convert_to_serializable(obj):
    """Convert numpy and tensorflow types to Python native types for JSON serialization"""
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (bytes, bytearray)):
        return obj.decode('utf-8')
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_to_serializable(i) for i in obj)
    elif hasattr(obj, 'numpy'):
        # For TensorFlow tensors
        return convert_to_serializable(obj.numpy())
    else:
        return obj

def save_history(history, filepath):
    """Save training history to a file"""
    # Convert history to a dictionary if it's a keras History object
    if not isinstance(history, dict):
        history = history.history

    # Convert all values to serializable types
    history_dict = convert_to_serializable(history)

    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(history_dict, f, indent=4)
    print(f"History saved to: {filepath}")

def save_metrics(metrics, filepath):
    """Save evaluation metrics to a file"""
    # Convert all values to serializable types
    serializable_metrics = convert_to_serializable(metrics)

    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(serializable_metrics, f, indent=4)
    print(f"Metrics saved to: {filepath}")

def load_metrics(filepath):
    """Load evaluation metrics from a file"""
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            metrics = json.load(f)
        return metrics
    return None

def finetune_and_evaluate_global_model(model, train_data_path, test_data_path,
                                       global_model_save_path, history_save_path,
                                       metrics_save_path, force_retrain=False):
    """Fine-tune and evaluate the global model, with option to load saved results"""

    # Check if model and results already exist
    if (os.path.exists(global_model_save_path) and
        os.path.exists(history_save_path) and
        os.path.exists(metrics_save_path) and
        not force_retrain):

        print(f"Global model already exists at {global_model_save_path}. Loading...")
        model = load_model(global_model_save_path)

        # Load metrics
        metrics = load_metrics(metrics_save_path)

        # Display metrics
        print("\nLoaded Global Model Test Metrics:")
        print("=" * 60)
        for metric_name, value in metrics.items():
            if metric_name != 'confusion_matrix' and metric_name != 'classification_report':
                print(f"{metric_name.capitalize()}: {value:.4f}")

        # Display confusion matrix if available
        if 'confusion_matrix' in metrics:
            plt.figure(figsize=(10, 8))
            cm = np.array(metrics['confusion_matrix'])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title('Global Model Confusion Matrix')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.show()

        # No need to retrain or reevaluate
        return model, None

    # If model doesn't exist or force_retrain is True, fine-tune and evaluate
    # Training data generator with augmentation
    train_datagen = ImageDataGenerator(
        rescale=1.0/255,
        rotation_range=30,
        width_shift_range=0.25,
        height_shift_range=0.25,
        shear_range=0.15,
        zoom_range=0.25,
        horizontal_flip=True,
        vertical_flip=False,
        fill_mode='nearest',
        validation_split=0.2
    )

    # Training generator
    train_generator = train_datagen.flow_from_directory(
        train_data_path,
        target_size=(224, 224),
        batch_size=16,
        class_mode="sparse",
        subset='training'
    )

    # Validation generator
    validation_generator = train_datagen.flow_from_directory(
        train_data_path,
        target_size=(224, 224),
        batch_size=16,
        class_mode="sparse",
        subset='validation'
    )

    # Only keeping ReduceLROnPlateau callback
    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=5,
            min_lr=1e-7
        )
    ]

    # Fine-tuning step
    print("Fine-tuning global model...")
    history = model.fit(
        train_generator,
        epochs=200,
        validation_data=validation_generator,
        callbacks=callbacks,
        workers=1,
        use_multiprocessing=False
    )

    # Test data evaluation
    test_datagen = ImageDataGenerator(rescale=1.0/255)
    test_generator = test_datagen.flow_from_directory(
        test_data_path,
        target_size=(224, 224),
        batch_size=32,
        class_mode="sparse",
        shuffle=False
    )

    # Final evaluation
    print("\nEvaluating on test data...")
    predictions = model.predict(test_generator)
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = test_generator.classes

    # Metrics calculation
    precision = precision_score(true_classes, pred_classes, average="weighted")
    recall = recall_score(true_classes, pred_classes, average="weighted")
    f1 = f1_score(true_classes, pred_classes, average="weighted")
    accuracy = accuracy_score(true_classes, pred_classes)

    # Create confusion matrix
    cm = confusion_matrix(true_classes, pred_classes)

    # Get classification report
    class_report = classification_report(true_classes, pred_classes)

    # Results visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Global Model Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    # Prepare metrics for display and saving
    metrics = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'confusion_matrix': cm.tolist(),
        'classification_report': class_report
    }

    print("\nGlobal Model Test Metrics:")
    print("=" * 60)
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

    # Save model
    os.makedirs(os.path.dirname(global_model_save_path), exist_ok=True)
    model.save(global_model_save_path)
    print(f"Global model saved to: {global_model_save_path}")

    # Save history
    save_history(history.history, history_save_path)

    # Save metrics
    save_metrics(metrics, metrics_save_path)

    return model, history

# Base directory for saved models and results
save_dir = "C:\\Users\\sa\\Desktop\\saved_models"
# Paths to saved client models
client1_model_path = os.path.join(save_dir, "client1_model.h5")
client2_model_path = os.path.join(save_dir, "client2_model.h5")
client3_model_path = os.path.join(save_dir, "client3_model.h5")
client4_model_path = os.path.join(save_dir, "client4_model.h5")
# Paths for global model and results
global_model_path = os.path.join(save_dir, "Global_models_fedavg.h5")
global_history_path = os.path.join(save_dir, "Global_history_fedavg.json")
global_metrics_path = os.path.join(save_dir, "Global_metrics_fedavg.json")
# Paths to data
train_data_path = "C:\\Users\\sa\\Desktop\\Balanced_Training1"
test_data_path = "C:\\Users\\sa\\Desktop\\Testing2"
# Set force_retrain to False to use saved global model if it exists
force_retrain = False

try:
    # Load saved client models
    print("Loading client models...")
    client_models = load_saved_models([
        client1_model_path,
        client2_model_path,
        client3_model_path,
        client4_model_path
    ])
    # Create global model by aggregating client models using FedAvg principles
    print("Creating global model by aggregating client models (FedAvg)...")
    global_model = aggregate_models_fedavg(client_models)
    # Fine-tune and evaluate global model
    print("Fine-tuning and evaluating global model...")
    global_model, history = finetune_and_evaluate_global_model(
        global_model,
        train_data_path,
        test_data_path,
        global_model_path,
        global_history_path,
        global_metrics_path,
        force_retrain=force_retrain
    )
    print("Global model processing completed successfully!")

except Exception as e:
    print(f"An error occurred: {str(e)}")
    import traceback
    traceback.print_exc()
